In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [2]:
#Extract text from pdf files 

def load_pdf_files(data):
    loader=DirectoryLoader(
        data, 
        glob="*.pdf", 
        loader_cls= PyMuPDFLoader
    )
    
    documents= loader.load()
    return documents

In [ ]:
extracted_data = load_pdf_files(data)


In [ ]:
extracted_data

In [ ]:
len(extracted_data)

In [10]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document])-> List[Document]:
    ## returns only the source na dthe original content
    print(len(docs))
    print(type(docs))
    
    minimal_docs: List[Document]=[]
    for doc in docs:
        src= doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content, 
                metadata={"source":src}
            )
        )
    return minimal_docs
    

In [ ]:
minimal_docs= filter_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs

In [ ]:
len(extracted_data)

In [ ]:
len(minimal_docs)

In [15]:
def text_split(minimal_docs):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size= 500,
        chunk_overlap=20
    )
    
    texts_chunk= text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [ ]:
text_chunk= text_split(minimal_docs)
print(len(text_chunk))

In [ ]:
text_chunk

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
def donwload_embeddings():
    ##download and return the hugging face embedding model
    
    model_name= "sentence-transformers/all-MiniLM-L6-v2"
    embeddings= HuggingFaceEmbeddings(
        model_name= model_name
    )
    
    return embeddings

embeddings= donwload_embeddings()

In [ ]:
embeddings

In [ ]:
vector= embeddings.embed_query("hello world")
vector

In [ ]:
len(vector)

In [ ]:
from dotenv import load_dotenv
import os 
load_dotenv(env)

In [23]:
PINECONE_API_KEY= os.getenv("PINECONE_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [24]:
from pinecone import Pinecone
pc=Pinecone(api_key=PINECONE_API_KEY)

In [ ]:
pc

In [28]:
from pinecone import ServerlessSpec

index_name="medical-chatbot"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name, 
        dimension= 384,   #dimension of embeddings
        metric= "cosine",   #cosine similarity
        spec= ServerlessSpec(cloud="aws", region="us-east-1")
    )
    
index= pc.Index(index_name)

In [25]:
#push records do not run again
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embeddings,
    index_name=index_name
)

In [26]:
#add more data to the existing pinecone index
dswith= Document(
    page_content= "it is good to take advice from a doctor when not sure!", 
                 metadata= {"source": "ethics"}
    )

In [29]:
#existing runnn 
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [ ]:
docsearch.add_documents(documents= [dswith])

In [30]:
retriever=docsearch.as_retriever(search_type= "similarity", search_kwargs={"k":3}) #return 3 releavmt responses

In [ ]:
retrieved_docs= retriever.invoke("what is acne?")
retrieved_docs

In [37]:
from langchain_groq import ChatGroq
chatmodel= ChatGroq(model="llama-3.1-8b-instant",
                                  temperature=0.3
                                  )

In [38]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


In [39]:
system_prompt=(
    "You are a medical assistant for questions-answering tasks. "
    "Use the following pieces of retirieved context to answer"
    "the question. If you do not know the answer, say that"
    "you don't know. Use three sentences maximum and keep the"
    "answer concise. "
    "\n\n"
    "{context}"
)

prompt= ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [40]:
question_answer_chain= create_stuff_documents_chain(chatmodel, prompt)
rag_chain= create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response= rag_chain.invoke({"input":"what is acne?"})
print(response["answer"])

In [ ]:
response=rag_chain.invoke({"input":"what is gigantism"})
print(response["answer"])